# StreamFlix Content Analytics Project

## Phase 3 – Business KPIs & Insights

**Analyst:** Hemanth

### Objective

The objective of this phase is to calculate key business performance indicators for StreamFlix using the cleaned datasets and interpret the results from a management perspective.

In [1]:
import pandas as pd
import numpy as np

In [2]:
watch_history = pd.read_csv(
    "../cleaned_data/watch_history_cleaned.csv",
    parse_dates=["watch_date"]
)

watch_history.shape

(650000, 10)

## KPI 1 – Total Watch Hours

Total Watch Hours measures the overall amount of time subscribers spent watching StreamFlix content.

In [3]:
total_watch_hours = (
    watch_history['watch_duration_min'].sum() / 60
)

print("Total Watch Hours:", round(total_watch_hours, 2))

Total Watch Hours: 3333467.73


In [4]:
subscribers = pd.read_csv(
    "../cleaned_data/subscribers_cleaned.csv",
    parse_dates=["signup_date", "churn_date"]
)

subscribers.shape

(15000, 14)

## KPI 2 – Active Subscriber Rate

Active Subscriber Rate measures the percentage of total subscribers who are currently active on StreamFlix.

In [5]:
active_subscribers = subscribers['is_active'].sum()

total_subscribers = len(subscribers)

active_rate = (
    active_subscribers / total_subscribers * 100
)

print("Active Subscribers:", active_subscribers)
print("Total Subscribers:", total_subscribers)
print("Active Subscriber Rate:", round(active_rate, 2), "%")

Active Subscribers: 11199
Total Subscribers: 15000
Active Subscriber Rate: 74.66 %


## KPI 3 – Churn Rate

Churn Rate measures the percentage of StreamFlix subscribers who have stopped their subscription and are no longer active.

In [6]:
churned_subscribers = (~subscribers['is_active']).sum()

churn_rate = (
    churned_subscribers / total_subscribers * 100
)

print("Churned Subscribers:", churned_subscribers)
print("Total Subscribers:", total_subscribers)
print("Churn Rate:", round(churn_rate, 2), "%")

Churned Subscribers: 3801
Total Subscribers: 15000
Churn Rate: 25.34 %


### Insight

Out of 15,000 StreamFlix subscribers, 3,801 have churned, resulting in a churn rate of 25.34%. Approximately 74.66% of subscribers remain active, indicating that about one-quarter of the subscriber base is no longer active.

## KPI 4 – Average Completion Rate

Average Completion Rate measures the average percentage of content watched across all StreamFlix viewing sessions.

In [7]:
average_completion_rate = watch_history['completion_pct'].mean()

print(
    "Average Completion Rate:",
    round(average_completion_rate, 2),
    "%"
)

Average Completion Rate: 65.31 %


## KPI 5 – Monthly Recurring Revenue (MRR)

Monthly Recurring Revenue estimates the total monthly subscription revenue generated by currently active StreamFlix subscribers.

In [8]:
active_customers = subscribers[
    subscribers['is_active'] == True
]

mrr = active_customers['monthly_price_usd'].sum()

print("Monthly Recurring Revenue (MRR): $", round(mrr, 2))

Monthly Recurring Revenue (MRR): $ 175868.51


## KPI 6 – Average Revenue Per User (ARPU)

ARPU measures the average monthly subscription revenue generated by each active StreamFlix subscriber.

In [9]:
arpu = mrr / active_subscribers

print("Average Revenue Per User (ARPU): $", round(arpu, 2))

Average Revenue Per User (ARPU): $ 15.7


## KPI 7 – Average Watch Time per Subscriber

Average Watch Time per Subscriber measures the average total number of hours watched by each StreamFlix subscriber.

In [10]:
average_watch_time_per_subscriber = (
    total_watch_hours / total_subscribers
)

print(
    "Average Watch Time per Subscriber:",
    round(average_watch_time_per_subscriber, 2),
    "hours"
)

Average Watch Time per Subscriber: 222.23 hours


In [11]:
watchlist = pd.read_csv(
    "../cleaned_data/watchlist_cleaned.csv",
    parse_dates=["added_date"]
)

watchlist.shape

(65000, 5)

## KPI 8 – Watchlist Conversion Rate

Watchlist Conversion Rate measures the percentage of saved titles that were later watched by subscribers.

In [12]:
total_watchlist_entries = len(watchlist)

watched_entries = watchlist['watched'].sum()

watchlist_conversion_rate = (
    watched_entries / total_watchlist_entries * 100
)

print("Total Watchlist Entries:", total_watchlist_entries)
print("Watched Entries:", watched_entries)
print(
    "Watchlist Conversion Rate:",
    round(watchlist_conversion_rate, 2),
    "%"
)

Total Watchlist Entries: 65000
Watched Entries: 30048
Watchlist Conversion Rate: 46.23 %


In [13]:
average_watch_time_per_subscriber = (
    total_watch_hours / active_subscribers
)

print(
    "Average Watch Time per Active Subscriber:",
    round(average_watch_time_per_subscriber, 2),
    "hours"
)

Average Watch Time per Active Subscriber: 297.66 hours


## KPI 9 – Hit Concentration

Hit Concentration measures the percentage of total viewing sessions generated by the top 10% most-played titles. A high concentration may indicate that platform engagement depends heavily on a relatively small portion of the catalogue.

In [14]:
title_plays = (
    watch_history
    .groupby('title_id')
    .size()
    .sort_values(ascending=False)
)

title_plays.head(10)

title_id
TTL205962    4436
TTL207141    4358
TTL203531    3482
TTL206337    3028
TTL207554    2476
TTL202855    1927
TTL204514    1792
TTL206938    1792
TTL204462    1681
TTL205320    1513
dtype: int64

In [15]:
top_10_percent_count = int(np.ceil(len(title_plays) * 0.10))

top_10_percent_plays = title_plays.head(
    top_10_percent_count
).sum()

total_plays = title_plays.sum()

hit_concentration = (
    top_10_percent_plays / total_plays * 100
)

print("Titles with viewing activity:", len(title_plays))
print("Top 10% Titles:", top_10_percent_count)
print("Total Plays:", total_plays)
print("Plays from Top 10% Titles:", top_10_percent_plays)
print("Hit Concentration:", round(hit_concentration, 2), "%")

Titles with viewing activity: 9000
Top 10% Titles: 900
Total Plays: 650000
Plays from Top 10% Titles: 201417
Hit Concentration: 30.99 %


In [16]:
titles = pd.read_csv(
    "../cleaned_data/titles_cleaned.csv",
    parse_dates=["date_added", "license_expiry"]
)

titles.shape

(9000, 21)

## KPI 10 – Originals Share of Watch Hours

Originals Share of Watch Hours measures the percentage of total StreamFlix watch time generated by StreamFlix Original content.

In [17]:
watch_originals = pd.merge(
    watch_history,
    titles[['title_id', 'is_original']],
    on='title_id',
    how='left'
)

In [18]:
original_watch_hours = (
    watch_originals.loc[
        watch_originals['is_original'] == True,
        'watch_duration_min'
    ].sum() / 60
)

originals_share = (
    original_watch_hours / total_watch_hours * 100
)

print("Original Watch Hours:", round(original_watch_hours, 2))
print("Total Watch Hours:", round(total_watch_hours, 2))
print("Originals Share of Watch Hours:", round(originals_share, 2), "%")

Original Watch Hours: 905661.77
Total Watch Hours: 3333467.73
Originals Share of Watch Hours: 27.17 %


## Phase 3 – KPI Summary

The following table summarizes the key business performance indicators calculated for StreamFlix.

In [19]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Watch Hours",
        "Active Subscriber Rate",
        "Churn Rate",
        "Average Completion Rate",
        "Monthly Recurring Revenue (MRR)",
        "Average Revenue Per User (ARPU)",
        "Average Watch Time per Active Subscriber",
        "Watchlist Conversion Rate",
        "Hit Concentration",
        "Originals Share of Watch Hours"
    ],
    
    "Value": [
        round(total_watch_hours, 2),
        round(active_rate, 2),
        round(churn_rate, 2),
        round(average_completion_rate, 2),
        round(mrr, 2),
        round(arpu, 2),
        round(average_watch_time_per_subscriber, 2),
        round(watchlist_conversion_rate, 2),
        round(hit_concentration, 2),
        round(originals_share, 2)
    ],
    
    "Unit": [
        "Hours",
        "%",
        "%",
        "%",
        "USD",
        "USD",
        "Hours",
        "%",
        "%",
        "%"
    ]
})

kpi_summary

,KPI,Value,Unit
0,Total Watch Hours,3333467.73,Hours
1,Active Subscriber Rate,74.66,%
2,Churn Rate,25.34,%
3,Average Completion Rate,65.31,%
4,Monthly Recurring Revenue (MRR),175868.51,USD
5,Average Revenue Per User (ARPU),15.70,USD
6,Average Watch Time per Active Subscriber,297.66,Hours
7,Watchlist Conversion Rate,46.23,%
8,Hit Concentration,30.99,%
9,Originals Share of Watch Hours,27.17,%


In [20]:
kpi_summary.to_csv(
    "../reports/Phase3_KPI_Summary_Hemanth.csv",
    index=False
)

print("KPI summary saved successfully.")

KPI summary saved successfully.


In [21]:
kpi_summary


,KPI,Value,Unit
0,Total Watch Hours,3333467.73,Hours
1,Active Subscriber Rate,74.66,%
2,Churn Rate,25.34,%
3,Average Completion Rate,65.31,%
4,Monthly Recurring Revenue (MRR),175868.51,USD
5,Average Revenue Per User (ARPU),15.70,USD
6,Average Watch Time per Active Subscriber,297.66,Hours
7,Watchlist Conversion Rate,46.23,%
8,Hit Concentration,30.99,%
9,Originals Share of Watch Hours,27.17,%


## Phase 3 – Key Business Insights

StreamFlix generated approximately 3.33 million total watch hours across the available dataset, showing substantial overall platform engagement.

The active subscriber rate is 74.66%, which is above the project target of 70%. The churn rate is 25.34%, remaining below the 30% threshold, although roughly one in four subscribers has still churned and retention should remain an important business focus.

The average content completion rate is 65.31%, exceeding the 60% target. This suggests that viewers generally watch a substantial portion of the content they start.

StreamFlix currently generates approximately $175,868.51 in Monthly Recurring Revenue from active subscribers, with an ARPU of $15.70 per active subscriber.

Average watch time is approximately 297.66 hours per active subscriber across the full analysis period. This represents cumulative engagement over the dataset timeline rather than monthly viewing time.

Watchlist conversion is 46.23%, exceeding the 40% target, meaning nearly half of saved titles eventually convert into viewing activity.

The top 10% of titles account for 30.99% of total plays. This shows some concentration around popular content, but viewing activity is not entirely dependent on a very small number of titles.

StreamFlix Originals contribute 27.17% of total watch hours, meaning approximately 72.83% of viewing time comes from non-Original content. This suggests licensed and other non-Original content currently drives the majority of engagement.